In [1]:
print("okay")

okay


In [2]:
from langchain_groq import ChatGroq


In [3]:
from dotenv import load_dotenv

In [4]:
import os

In [5]:
load_dotenv()

True

In [6]:
#os.getenv("GROQ_API_KEY")

In [7]:
#os.getenv("GOOGLE_API_KEY")

In [8]:
llm=ChatGroq(model="groq/compound-mini")

In [9]:
print(llm.invoke("What is the capital of France?").content) 

Paris.


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [11]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [12]:
embedding_model.embed_query("What is the capital of France?")

[-0.032554302,
 0.01305496,
 0.015067407,
 -0.07128718,
 -0.031062366,
 -0.019940834,
 -0.013077609,
 -0.009117966,
 0.038935922,
 -0.01930863,
 -0.037332542,
 0.0015812494,
 -0.008252243,
 -0.012736891,
 0.12141348,
 -0.008084396,
 -0.00675194,
 0.0136525165,
 -0.007993852,
 -0.011459375,
 -0.015845438,
 0.0054860776,
 0.0081860535,
 -0.00075686385,
 -0.0036939767,
 0.022143796,
 -0.005230535,
 0.0029376769,
 0.0068194214,
 0.043721262,
 0.005832777,
 -0.0121669825,
 -0.0052677128,
 -0.0039394493,
 0.02664272,
 -0.005785547,
 0.0083369315,
 -0.018645173,
 -0.005101227,
 0.0016109659,
 -0.0016082282,
 -0.023501702,
 0.024029724,
 -0.0023990977,
 0.0036964675,
 -0.005751388,
 0.0014105918,
 -0.039790608,
 -0.025969282,
 0.04065211,
 -0.008603448,
 0.009018259,
 0.0074516605,
 -0.17227986,
 7.415835e-05,
 0.01012602,
 0.0033420047,
 -0.009170429,
 0.023084901,
 -0.029130757,
 -0.019378372,
 0.004348516,
 0.0025302735,
 -0.020455236,
 0.019821439,
 0.02472824,
 0.004933409,
 0.031398825,


# Data Ingestion


In [13]:
from langchain_community.document_loaders import PyPDFLoader

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
import os
os.getcwd()

'c:\\Users\\91984\\chat_portal_project\\notebook'

In [16]:
file_path = os.path.join(os.getcwd(), "data", "sample.pdf")
#"C:/Users/91984/chat_portal_project/notebook/sample.pdf"

In [17]:
loader = PyPDFLoader(file_path)

In [18]:
documents = loader.load() #loading the document 

In [19]:
len(documents) #number of pages in the pdf

77

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=120, 
    length_function=len  
)

In [21]:
docs = text_splitter.split_documents(documents)

In [22]:
docs[0].metadata #metadata of the first page i.e data about the data

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-07-20T00:30:36+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-07-20T00:30:36+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'c:\\Users\\91984\\chat_portal_project\\notebook\\data\\sample.pdf',
 'total_pages': 77,
 'page': 0,
 'page_label': '1'}

In [23]:
docs[0].page_content #content of the first page of the pdf

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev'

In [24]:
len(docs) #number of chunks the document is split into

738

In [25]:
from langchain_community.vectorstores import FAISS

In [26]:
import time

In [27]:
#time.sleep(60)

In [28]:
#embedding_model.embed_documents(docs[0].page_content) #embedding the document

In [29]:
time.sleep(120)

In [ ]:
import time

def get_all_embeddings_safely(docs, batch_size=20, delay=90.0):
    """
    Processes documents in batches to avoid 429 Resource Exhausted errors.
    - docs: Your list of document objects
    - batch_size: How many docs to send in one API call (max 100 for Free Tier)
    - delay: Seconds to wait between batches to stay under the 100 RPM limit
    """
    all_embeddings = []
    total_docs = len(docs)
    
    print(f"Starting embedding for {total_docs} documents...")

    for i in range(0, total_docs, batch_size):
        # 1. Grab a 'chunk' of documents
        current_batch = docs[i : i + batch_size]
        
        # 2. Fix the AttributeError: Extract text from each doc in the batch
        batch_texts = [doc.page_content for doc in current_batch]
        
        try:
            # 3. Call the embedding model (1 batch = 1 API request)
            embeddings = embedding_model.embed_documents(batch_texts)
            all_embeddings.extend(embeddings)
            
            print(f"Progress: {min(i + batch_size, total_docs)}/{total_docs} done.")
            
            # 4. Rate Limit: Pause to keep the "Traffic Cop" happy
            if i + batch_size < total_docs:
                time.sleep(delay)
                
        except Exception as e:
            print(f"Error at index {i}: {e}")
            print("Sleeping for 240 seconds to reset quota...s")
            time.sleep(240)  # Sleep for 4 minutes before retrying the next batch (adjust as needed)
            # Optional: If required Add retry logic here
            
    return all_embeddings

# --- HOW TO RUN IT ---
# This will handle the docs[0:2] you were trying, or even the whole list!
final_embeddings = get_all_embeddings_safely(docs)

Starting embedding for 738 documents...
Error at index 0: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 54.374321817s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUse

In [ ]:
len(final_embeddings)

In [ ]:
doc_vector= final_embeddings.copy()

In [ ]:
#time.sleep(65)

In [ ]:
#all_contents = [doc.page_content for doc in docs[:len(docs)]] 
#embeddings = embedding_model.embed_documents(all_contents)

In [ ]:
docs

In [ ]:
len(docs)

In [ ]:
vectorstore=FAISS.from_documents(docs, embedding_model)

In [ ]:
#Retrieval process
relavent_doc = vectorstore.similarity_search("llama2 finetuning benchmark experiments")

In [ ]:
relavent_doc

In [ ]:
relavent_doc[0].page_content

In [ ]:
relavent_doc[1].page_content

# Retrieval Process

In [ ]:
relavent_doc = vectorstore.similarity_search("llama2 finetuning benchmark experiments", k=10)

In [ ]:
relavent_doc

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
prompt_template = """Use the following pieces of context to answer the question at the end. 
                    If you don't know the answer, say you don't know, don't try to make up an answer.
                    context: {context} 
                    Question: {question}
                    Answer:"""


In [ ]:
prompt_template

In [ ]:
#from langchain_prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate


In [ ]:
prompt = PromptTemplate(
    template = prompt_template,
    input_variables=["context", "question"]
)

In [ ]:
prompt

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
Parser = StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
rag_chain = (
    {"context": retriever | format_docs , "question": RunnablePassthrough() }
    | prompt 
    | llm 
    | StrOutputParserarser 
)

In [ ]:
rag_chain.invoke("llama2 finetuning benchmark experiments")